# Race deep dive: one random race from the Betfair stream recordings

Picks a random recording from `My Drive/betfair stream data/recordings` and
looks at how that WIN market traded in the ~10 minutes before the off:

1. race facts: venue, type, field, commission, scheduled vs actual off, result, BSP
2. per-runner summary: opening, scheduled-start and off prices, moves, volume, spread, weight of money
3. market-level behaviour by phase: volume, overround, spread
4. **Chart 1: traded volume over time**
5. **Chart 2: price over time**, per runner
6. **Chart 3: relative price over time**: each runner's share of the market (normalised implied probability)
7. co-movement between runners, and whether weight of money predicted the next move in this race
8. auto-generated observations

Set `RACE` to a filename fragment to pin a race, or `SEED` to make the random
pick reproducible. Re-run from the "Pick a race" cell to draw another one.
All times are seconds (or minutes) relative to the **scheduled** start; the
dashed line marks the scheduled start, the solid red line marks the actual off (in-play).

In [ ]:
import os, sys, glob, random, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/betfair stream data'
    REPO = '/content/AHR_RL'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '-b', 'claude/wizardly-clarke-rb37hl',
                        'https://github.com/tristanhowells/AHR_RL.git', REPO], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'orjson'], check=False)
else:  # running from a local checkout: use the sample data in <repo>/data
    REPO = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DRIVE = os.path.join(REPO, 'data')
sys.path.insert(0, REPO)
REC_DIR = f'{DRIVE}/recordings'
CAT_DIR = f'{DRIVE}/catalogues'   # optional: runner names and form
print('recordings:', REC_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timezone

from ahr_rl.stream import read_recording
from ahr_rl.tape import build_tape
from ahr_rl.ladder import PRICES, N_TICKS

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

## Pick a race

In [ ]:
RACE = None   # e.g. 'Melton_1_262864890' to pin a race; None = random
SEED = None   # int for a reproducible random pick
MIN_STEPS = 600  # skip recordings with < 5 minutes of pre-race data (truncated / abandoned)

files = sorted(glob.glob(f'{REC_DIR}/**/*.ndjson.gz', recursive=True))
print(f'{len(files)} recordings found')
rng = random.Random(SEED)
candidates = [f for f in files if RACE in f] if RACE else rng.sample(files, len(files))
if not candidates:
    raise ValueError(f'no recording matches {RACE!r}')
tape = None
for path in candidates:
    t = build_tape(path)
    if t is not None and t.n_steps >= MIN_STEPS and t.n_runners >= 2:
        tape = t
        break
    print('skipping (too short / empty):', os.path.basename(path))
if tape is None:
    raise RuntimeError('no usable recording found')
rec = read_recording(path)
print('selected:', os.path.basename(path))

## 1. Race facts

In [ ]:
# market definition: first and last seen before the off
defs = [mc['marketDefinition'] for m in rec.messages for mc in m['mc'] if 'marketDefinition' in mc]
md0, md1 = defs[0], [d for d in defs if not d.get('inPlay')][-1] if any(not d.get('inPlay') for d in defs) else defs[-1]

# runner names from the catalogue if it was recorded, else "#<sort priority>"
names = {}
try:
    from ahr_rl.catalogue import index_catalogues, load_catalogue
    cp = index_catalogues(CAT_DIR).get(tape.market_id) if os.path.isdir(CAT_DIR) else None
    cat = load_catalogue(cp) if cp else None
    if cat:
        names = {int(r['selectionId']): r.get('runnerName', '') for r in cat.get('runners', [])}
except Exception as e:  # names are cosmetic
    cat = None
    print('catalogue not used:', e)
prio = {r['id']: r.get('sortPriority', i + 1) for i, r in enumerate(md0['runners'])}
R = tape.n_runners
sids = tape.selection_ids
label = [names.get(int(s)) or f'#{prio.get(int(s), i + 1)} ({int(s)})' for i, s in enumerate(sids)]

sched = datetime.fromtimestamp(rec.market_start_ms / 1000, tz=timezone.utc)
trailer = rec.trailer or {}
off_s = (trailer['inplay_from_pt'] - rec.market_start_ms) / 1000 if trailer.get('inplay_from_pt') else \
        (float(tape.t_rel[-1]) if tape.went_in_play else np.nan)
pre_window = [label[r] for r in range(R) if not tape.active.any(0)[r]]           # scratched before the recording
removed = [label[r] for r in range(R) if tape.active.any(0)[r] and not tape.active[-1, r]]
facts = {
    'market': tape.market_id,
    'venue': md1.get('venue'),
    'race type': md1.get('raceType', 'n/a'),
    'market name': (cat or {}).get('marketName', 'n/a'),
    'scheduled start (UTC)': sched.strftime('%Y-%m-%d %H:%M'),
    'local timezone': md1.get('timezone'),
    'runners (start of recording)': R,
    'scratched before window': ', '.join(pre_window) or 'none',
    'scratched during window': ', '.join(removed) or 'none',
    'commission (base rate %)': tape.base_rate,
    'recording starts (s vs scheduled)': round(float(tape.t_rel[0]), 1),
    'went in-play': tape.went_in_play,
    'actual off (s after scheduled)': round(off_s, 1) if np.isfinite(off_s) else 'n/a',
    'total matched at off ($, single-counted)': round(float(tape.total_matched[-1]), 0),
    'winner': label[tape.winner] if tape.winner >= 0 else 'unknown (recording ended before settlement)',
}
pd.Series(facts, name='value').to_frame()

## 2. Per-runner time series

Built from the 0.5s tape (the same grid the RL environment uses).

* The analysis stops at the last snapshot before the market suspended for the off; "off" prices below are that snapshot.
* **price**: midpoint of best back and best lay in *tick* space. If one side is missing it falls back to the last traded price.
* **market share**: 1/price for that runner divided by the sum across active runners. This is the "relative price" in Chart 3, and it removes the overround.
* **WoM** (weight of money): back-side size ÷ (back + lay size) over the top 3 levels.

In [ ]:
# keep only the pre-off book: drop the suspension / in-play rows at the end
ok = ~tape.suspended
if np.isfinite(off_s):
    ok &= tape.t_rel < off_s
T = int(np.where(ok)[0].max()) + 1
t = tape.t_rel[:T].astype(float)
tmin = t / 60
act = tape.active[:T].copy()
runners = [r for r in range(R) if act[:, r].any()]   # runners that were live at some point in the window

def tick_price(ticks):
    ticks = np.asarray(ticks, float)
    out = np.interp(np.clip(ticks, 0, N_TICKS - 1), np.arange(N_TICKS), PRICES)
    return np.where(ticks >= 0, out, np.nan)

bt, lt = tape.back_tick[:T, :, 0].astype(float), tape.lay_tick[:T, :, 0].astype(float)
bt[bt < 0], lt[lt < 0] = np.nan, np.nan
ltp = tick_price(tape.ltp_tick[:T].astype(float))
mid_tick = (bt + lt) / 2
price = np.where(np.isfinite(mid_tick), tick_price(np.nan_to_num(mid_tick, nan=-1)), ltp)
price[~act] = np.nan
spread = lt - bt
spread[~act] = np.nan
bsz, lsz = tape.back_size[:T, :, :3].sum(-1), tape.lay_size[:T, :, :3].sum(-1)
with np.errstate(invalid='ignore', divide='ignore'):
    wom = np.where(bsz + lsz > 0, bsz / (bsz + lsz), np.nan)
wom[~act] = np.nan
ip = 1 / price
share = ip / np.nansum(ip, axis=1, keepdims=True) * 100
overround_back = np.nansum(1 / tick_price(np.nan_to_num(bt, nan=-1)) * act, axis=1) * 100
overround_lay = np.nansum(1 / tick_price(np.nan_to_num(lt, nan=-1)) * act, axis=1) * 100

# trades (single-counted $) per runner
keep = tape.trade_step < T
tr = pd.DataFrame({'step': tape.trade_step[keep], 'runner': tape.trade_runner[keep],
                   'price': PRICES[tape.trade_tick[keep]], 'vol': tape.trade_vol[keep]})
tr['t'] = t[tr['step']]

def at(arr, sec):
    '''value of [T, R] array at the grid row closest to `sec`, or nan if outside the window'''
    if sec < t[0] - 1 or sec > t[-1] + 1:
        return np.full(arr.shape[1], np.nan)
    return arr[int(np.abs(t - sec).argmin())]

def first_valid(arr):
    out = np.full(arr.shape[1], np.nan)
    for r in range(arr.shape[1]):
        v = arr[:, r][np.isfinite(arr[:, r])]
        if len(v):
            out[r] = v[0]
    return out

def last_valid(arr):
    return first_valid(arr[::-1])

p_open, p_off = first_valid(price), last_valid(price)
tick_open = np.array([np.nan if not np.isfinite(p) else np.abs(PRICES - p).argmin() for p in p_open])
tick_off = np.array([np.nan if not np.isfinite(p) else np.abs(PRICES - p).argmin() for p in p_off])

### Runner summary (sorted by price at the off, favourite first)

In [ ]:
vol = tr.groupby('runner')['vol'].sum().reindex(range(R), fill_value=0)
ntr = tr.groupby('runner').size().reindex(range(R), fill_value=0)
vwap = (tr.assign(pv=tr['price'] * tr['vol']).groupby('runner')['pv'].sum() / tr.groupby('runner')['vol'].sum()).reindex(range(R))
pmin = tr.groupby('runner')['price'].min().reindex(range(R))
pmax = tr.groupby('runner')['price'].max().reindex(range(R))
s_open, s_off = first_valid(share), last_valid(share)
rel_chg = (s_off / s_open - 1) * 100

def verdict(x):
    if not np.isfinite(x):
        return ''
    return 'STEAMER' if x > 15 else 'firmer' if x > 5 else 'DRIFTER' if x < -15 else 'weaker' if x < -5 else 'steady'

summary = pd.DataFrame({
    'runner': label,
    'open': p_open,
    'T-5m': at(price, -300),
    'T-1m': at(price, -60),
    'sched start': at(price, 0),
    'off': p_off,
    'BSP': np.where(tape.bsp > 0, tape.bsp, np.nan),
    'ticks moved': tick_off - tick_open,          # negative = shortened (backed in)
    'mkt share open %': s_open,
    'mkt share off %': s_off,
    'share chg %': rel_chg,
    'move': [verdict(x) for x in rel_chg],
    'matched $': vol.values,
    'vol share %': vol.values / max(vol.sum(), 1e-9) * 100,
    'trades': ntr.values,
    'VWAP': vwap.values,
    'traded range': [f'{a:.2f}-{b:.2f}' if np.isfinite(a) else '' for a, b in zip(pmin, pmax)],
    'avg spread (ticks)': [np.nanmean(spread[:, r]) if r in runners else np.nan for r in range(R)],
    'avg WoM': [np.nanmean(wom[:, r]) if r in runners else np.nan for r in range(R)],
    'result': ['WON' if r == tape.winner else ('scratched' if not tape.active[-1, r] else '') for r in range(R)],
})
order = np.array(sorted(runners, key=lambda r: p_off[r] if np.isfinite(p_off[r]) else 1e9))
summary = summary.iloc[order].reset_index(drop=True)
summary.index += 1
summary.round(2)

## 3. Market-level behaviour by phase

In [ ]:
edges = [-np.inf, -300, -120, -60, 0, np.inf]
names_ph = ['open → T-5m', 'T-5m → T-2m', 'T-2m → T-1m', 'T-1m → sched start', 'sched start → off']
tr['phase'] = pd.cut(tr['t'], edges, labels=names_ph, right=False)
row_phase = pd.cut(pd.Series(t), edges, labels=names_ph, right=False)
fav = int(order[0])
phase = pd.DataFrame({
    'seconds': row_phase.value_counts(sort=False) * tape.dt,
    'matched $': tr.groupby('phase', observed=False)['vol'].sum(),
    'trades': tr.groupby('phase', observed=False).size(),
    'overround back %': pd.Series(overround_back).groupby(row_phase, observed=False).mean(),
    'overround lay %': pd.Series(overround_lay).groupby(row_phase, observed=False).mean(),
    'fav spread (ticks)': pd.Series(spread[:, fav]).groupby(row_phase, observed=False).mean(),
    'fav best-level $ (back+lay)': pd.Series(tape.back_size[:T, fav, 0] + tape.lay_size[:T, fav, 0]).groupby(row_phase, observed=False).mean(),
})
phase['vol share %'] = phase['matched $'] / max(phase['matched $'].sum(), 1e-9) * 100
phase['$ per minute'] = phase['matched $'] / (phase['seconds'] / 60).replace(0, np.nan)
phase.round(1)

## Charts

Runners keep the same colour in all three charts, ordered favourite first. ★ marks the winner.

In [ ]:
cmap = plt.get_cmap('tab10' if R <= 10 else 'tab20')
color = {int(r): cmap(i % cmap.N) for i, r in enumerate(order)}
leg = {r: f"{label[r]}{' ★' if r == tape.winner else ''}" + (f"  BSP {tape.bsp[r]:.2f}" if tape.bsp[r] > 0 else '')
       for r in range(R)}

def mark_times(ax):
    ax.axvline(0, color='k', ls='--', lw=1, label='scheduled start')
    if np.isfinite(off_s):
        ax.axvline(off_s / 60, color='red', lw=1.2, label=f'off (+{off_s:.0f}s)')
    ax.set_xlabel('minutes relative to scheduled start')
    ax.set_xlim(tmin[0], tmin[-1] + 0.1)

title = f"{facts['venue']} {facts['race type']} | {facts['scheduled start (UTC)']} UTC | {tape.market_id}"

### Chart 1: traded volume over time

Bars: dollars matched per 15-second bucket, stacked by runner. Line (right axis):
cumulative market total matched. Volumes are single-counted: the raw `trd` field
double-counts, so it is halved.

In [ ]:
BUCKET = 15  # seconds
bins = np.arange(np.floor(t[0] / BUCKET) * BUCKET, t[-1] + BUCKET, BUCKET)
fig, ax = plt.subplots(figsize=(13, 5.5))
bottom = np.zeros(len(bins) - 1)
for r in order:
    v, _ = np.histogram(tr.loc[tr['runner'] == r, 't'], bins=bins, weights=tr.loc[tr['runner'] == r, 'vol'])
    ax.bar(bins[:-1] / 60, v, width=BUCKET / 60, align='edge', bottom=bottom, color=color[int(r)],
           label=leg[r], edgecolor='none')
    bottom += v
ax.set_ylabel(f'$ matched per {BUCKET}s')
ax2 = ax.twinx()
ax2.plot(tmin, tape.total_matched[:T], color='k', lw=1.8, label='cumulative total matched')
ax2.set_ylabel('cumulative $ matched')
ax2.grid(False)
ax2.set_ylim(0, None)
mark_times(ax)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8, ncol=2)
ax.set_title(f'Traded volume over time: {title}')
plt.tight_layout(); plt.show()

### Chart 2: price over time

Mid price per runner, in decimal odds, on a log scale so a tick move looks
about the same size at 2.0 and at 20.0. Dots mark scratched runners' last price.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.5))
for r in order:
    ax.plot(tmin, price[:, r], color=color[int(r)], lw=2.2 if r == tape.winner else 1.3, label=leg[r])
    if not act[-1, r]:
        k = np.where(np.isfinite(price[:, r]))[0]
        if len(k):
            ax.plot(tmin[k[-1]], price[k[-1], r], 'o', color=color[int(r)])
ax.set_yscale('log')
ax.yaxis.set_major_locator(mticker.FixedLocator([1.5, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 100, 200, 500, 1000]))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%g'))
ax.yaxis.set_minor_formatter(mticker.NullFormatter())
ax.set_ylabel('price (decimal odds, log scale)')
mark_times(ax)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
ax.set_title(f'Price over time: {title}')
plt.tight_layout(); plt.show()

### Chart 3: relative price over time (share of the market)

Each runner's implied probability as a share of the whole field: 1/price,
normalised across active runners so the shares sum to 100%. This separates a
runner genuinely firming or drifting *against the rest of the field* from
the whole book tightening (overround falling) as liquidity arrives. The lower
panel shows each runner's share relative to its own opening share.

In [ ]:
fig, (ax, axb) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={'height_ratios': [3, 2]})
for r in order:
    ax.plot(tmin, share[:, r], color=color[int(r)], lw=2.2 if r == tape.winner else 1.3, label=leg[r])
    axb.plot(tmin, (share[:, r] / s_open[r] - 1) * 100, color=color[int(r)], lw=2.2 if r == tape.winner else 1.1)
ax.set_ylabel('share of market (%)')
ax.set_ylim(0, None)
ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
ax.set_title(f'Relative price over time: {title}')
axb.axhline(0, color='grey', lw=0.8)
axb.set_ylabel('change vs opening share (%)\n(+ = firming, − = drifting)')
mark_times(ax); mark_times(axb)
ax.set_xlabel('')
plt.tight_layout(); plt.show()

## 4. Co-movement and weight of money in this race

* **Correlation of 30-second moves** (log price changes) between the top runners.
  Money leaving one horse usually goes to others, so you'd expect negative
  values if the market re-prices the field as a whole.
* **WoM → next 30s move**: correlation between WoM (share of resting money on the
  back side, i.e. offered by layers) and the next 30s log price change. Positive
  means that when more money was waiting to be backed, the price drifted next.
  It's one race, so treat it as colour, not evidence. Across the dataset, WoM was
  the strongest single signal (|correlation| ≈ 0.18, same sign in 99% of races)
  but too weak to beat trading costs.

In [ ]:
TOP = min(6, R)
top = [int(r) for r in order[:TOP]]
k30 = int(30 / tape.dt)
lp = pd.DataFrame(np.log(price[:, top]), columns=[label[r] for r in top]).iloc[::k30]
corr = lp.diff().corr()
display(corr.round(2)) if 'display' in globals() else print(corr.round(2))

rows = []
for r in runners:
    fwd = np.log(np.roll(price[:, r], -k30) / price[:, r])
    fwd[-k30:] = np.nan
    w = wom[:, r] - 0.5
    m = np.isfinite(fwd) & np.isfinite(w) & (np.arange(T) % 20 == 0)   # every 10s to limit overlap
    if m.sum() > 10 and np.std(w[m]) > 0 and np.std(fwd[m]) > 0:
        rows.append(dict(runner=label[r], obs=int(m.sum()), corr_wom_next30=np.corrcoef(w[m], fwd[m])[0, 1]))
wom_df = pd.DataFrame(rows).sort_values('corr_wom_next30', ascending=False)
print('positive = more money waiting to be backed was followed by a drift')
wom_df.round(3)

## 5. Observations (generated from the numbers above)

In [ ]:
s = summary
obs = []
fav_open = s.loc[s['open'].idxmin(), 'runner']
fav_off = s.iloc[0]['runner']
obs.append(f"Favourite at the open: {fav_open} ({s['open'].min():.2f}). Favourite at the off: {fav_off} ({s.iloc[0]['off']:.2f})."
           + (' Favouritism changed hands.' if fav_open != fav_off else ''))
if tape.winner >= 0:
    wrow = s[s['result'] == 'WON'].iloc[0]
    obs.append(f"Winner: {wrow['runner']}, {wrow.name}{'st' if wrow.name == 1 else 'nd' if wrow.name == 2 else 'rd' if wrow.name == 3 else 'th'} in the market at the off "
               f"({wrow['off']:.2f}, BSP {wrow['BSP']:.2f}); its market share moved {wrow['share chg %']:+.1f}% ({wrow['move'] or 'n/a'}).")
st = s[s['share chg %'].notna()]
if len(st):
    a, b = st.loc[st['share chg %'].idxmax()], st.loc[st['share chg %'].idxmin()]
    obs.append(f"Biggest steamer: {a['runner']} {a['open']:.2f} → {a['off']:.2f} ({a['ticks moved']:+.0f} ticks, share {a['share chg %']:+.1f}%).")
    obs.append(f"Biggest drifter: {b['runner']} {b['open']:.2f} → {b['off']:.2f} ({b['ticks moved']:+.0f} ticks, share {b['share chg %']:+.1f}%).")
tot = tr['vol'].sum()
late = tr.loc[tr['t'] >= -60, 'vol'].sum()
obs.append(f"${tot:,.0f} matched in the window; {late / max(tot, 1e-9) * 100:.0f}% of it in the last minute before the scheduled start or later. "
           f"Top 3 runners took {s['vol share %'].iloc[:3].sum():.0f}% of the volume.")
obs.append(f"Back-side overround went from {overround_back[np.isfinite(overround_back)][0]:.1f}% at the open to {overround_back[-1]:.1f}% just before the off "
           f"(100% = a perfectly efficient book).")
obs.append(f"Favourite's average spread: {np.nanmean(spread[:, fav]):.1f} ticks; average best-level size (back+lay) "
           f"${np.nanmean(tape.back_size[:T, fav, 0] + tape.lay_size[:T, fav, 0]):,.0f}.")
if np.isfinite(off_s):
    obs.append(f"The off came {off_s:+.0f}s after the scheduled start (dataset median ≈ +68s).")
if removed:
    obs.append(f"Scratched during the window: {', '.join(removed)}.")
if len(wom_df):
    c = wom_df['corr_wom_next30']
    obs.append(f"WoM → next-30s move correlation: median {c.median():+.2f} across runners, positive for {(c > 0).sum()} of {len(c)} "
               "(positive = money waiting to be backed preceded a drift).")
print('\n'.join('• ' + o for o in obs))